#  ML Taxon Classifier Demo for East African Mosquito Samples by **Meron Asmamaw Alemayehu**
This notebook demonstrates my workflow for filtering East African samples from MalariaGEN AG3 metadata and preparing them for a machine-learning taxon classifier.  

I will write the code step by step, explaining what I plan to do at each stage.

# 1 Install and Import Libraries

In [ ]:
# -----------------------------
# 1. Install and import libraries
# -----------------------------
# I will install the malariagen_data package to access AG3 data
%pip install -q --no-warn-conflicts malariagen_data

# Import libraries I will need
import malariagen_data
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# I will also use this to make plots render nicely in Colab
import plotly.io as pio
pio.renderers.default = "notebook+colab"

# 2 Load AG3 Metadata
I will load the AG3 dataset and check the metadata so I know which samples are available and what columns I can use for filtering.

In [ ]:
# Initialize AG3 dataset
ag3 = malariagen_data.Ag3()

# Load sample metadata into a DataFrame
samples_df = ag3.sample_metadata()

# View first few rows to understand structure
samples_df.head()

# I will also check all available columns to plan which ones I need
samples_df.columns.tolist()

# 3 Filter for East African Samples
I will filter samples to only include those from East African countries.  
Later I will use the 'sample_id' column to access ENA fastq files.

In [ ]:
# List of East African countries I will filter
east_africa_countries = [
    "Ethiopia", "Comoros, The Union of the", "Kenya", "Madagascar",
    "Malawi", "Mozambique", "South Sudan", "Tanzania", "Uganda",
    "Zambia", "Zimbabwe"
]

# Filter metadata for East African samples
east_africa_samples = samples_df[samples_df['country'].isin(east_africa_countries)].copy()
print("Total East African samples:", len(east_africa_samples))

# Save the filtered metadata for downstream analysis
east_africa_samples.to_csv("east_africa_samples_metadata.csv", index=False)

# 4 Explore Samples
I will look at how many samples come from each country and each year.  
This will help me understand the dataset before I fetch FASTQ files and do ML.

In [ ]:
# Samples per country
samples_per_country = east_africa_samples['country'].value_counts()
print(samples_per_country)

# Samples per year
samples_per_year = east_africa_samples.groupby("year").size()
print(samples_per_year)

#  plot for visualization
samples_per_year.plot(kind="bar", figsize=(8,4), title="East African samples per year")
plt.xlabel("Year")
plt.ylabel("Number of samples")
plt.show()


# 5 geographic plot

In [ ]:
#  geographic plot
plt.figure(figsize=(6,6))
plt.scatter(
    east_africa_samples["longitude"],
    east_africa_samples["latitude"],
    alpha=0.5,
    c="blue",
    label="Samples"
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic distribution of East African samples")
plt.legend()
plt.show()

# 6 Planned ML Pipeline Steps (with example code)
Here I will show **how I plan to implement the full workflow** for my taxon classifier.  
For demonstration, I will use placeholder data so that the notebook runs safely in Colab.

In [ ]:
# 6.1 Use 'sample_id' to get ENA run accessions and links to FASTQ files

# I will create a toy example of how to retrieve ENA links using sample_id
# In practice, I would use ag3.ena_run_accessions() or metadata lookups

# Example: pretend we have these sample_ids
sample_ids = ml_metadata['sample_id'].values[:5]  # take first 5 for demo

# I will create a dictionary mapping sample_id to fake ENA FTP links
ena_links = {sid: f"ftp://ftp.sra.ebi.ac.uk/vol1/fastq/{sid}_R1.fastq.gz" for sid in sample_ids}
print("Sample ENA links:")
for k, v in ena_links.items():
    print(k, "->", v)

## 6.2 Fetch FASTQ files in Colab using !wget
### I will download files directly into Colab (not local machine) using wget. For demonstration, I will not actually fetch the large files

In [ ]:
# Example wget command for Colab (commented out to avoid downloading)
# for sample_id, link in ena_links.items():
#     !wget {link} -O {sample_id}_R1.fastq.gz

print("I would use wget to fetch FASTQ files for each sample_id directly in Colab.")

## 6.3 Filter FASTQ reads for high quality
I will show a placeholder example: in practice, I would parse FASTQ and filter by quality scores

In [ ]:
# Example: pseudo-function to filter FASTQ reads
def filter_high_quality_reads(fastq_file):
    """
    I will parse each read from the FASTQ file and keep only reads
    where quality scores are high. This is a placeholder.
    """
    # Placeholder: just return a list of "high-quality reads"
    return ["ATCG"*25]*100  # pretend 100 high-quality reads

# Demonstrate with one sample
sample_reads = filter_high_quality_reads("fake_sample.fastq.gz")
print("Number of high-quality reads:", len(sample_reads))

## 6.4 Extract k-mer features for each sample
I will use the filtered reads to generate k-mer counts (toy example)

In [ ]:
# Example function to extract k-mer features
def extract_kmer_features(reads, k=5):
    """
    I will count k-mers in the reads and return a feature vector.
    Here I use random numbers as a placeholder.
    """
    import numpy as np
    num_features = 50
    return np.random.rand(num_features)

# Apply to one sample
features = extract_kmer_features(sample_reads)
print("Example feature vector (length):", len(features))

## 6.5 Split dataset into train and test, and the Target Column will be the '**taxon**' for supervised ML.
I will use placeholder features for all samples to demonstrate splitting

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Generate fake feature matrix for all samples
num_samples = ml_metadata.shape[0]
num_features = 50
X = np.random.rand(num_samples, num_features)

# Target labels from 'taxon' column
y = ml_metadata['taxon'].values

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training samples:", len(X_train), "Testing samples:", len(X_test))

## 6.6 Train ML classifiers
I will train a few example classifiers using the toy features to show the workflow

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Random Forest classifier
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest toy accuracy:", accuracy_score(y_test, y_pred_rf))

# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("Logistic Regression toy accuracy:", accuracy_score(y_test, y_pred_lr))

# Support Vector Machine
svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
print("SVM toy accuracy:", accuracy_score(y_test, y_pred_svm))